# PowSyBl Load Flow

**Hands-On Power System Analysis with PyPowSyBl**  
GridFM Workshop — Harvard, March 2026

---

## Setup

Run the cell below to install the required packages (only needed once on Colab).

In [ ]:
!pip install pypowsybl pypowsybl-jupyter matplotlib pandas numpy

### Import helper functions

The `helpers.py` file contains pre-configured network loaders and utility functions.

In [ ]:
from helpers import *

---

## Power Flow Algorithms and Configuration Parameters

PyPowSyBl delegates power flow computation to **solver providers**. The default provider is **OpenLoadFlow**, an open-source Java-based solver developed alongside PowSyBl.

### Available Load Flow Providers

**Task:** List the available load flow providers and the default provider using `lf.get_provider_names()` and `lf.get_default_provider()`.

In [ ]:
# Hint: lf.get_provider_names(), lf.get_default_provider()


**Task:** Retrieve the full parameter metadata for the default provider using `lf.get_provider_parameters()`. Group by `category_key` to see all settings organized by category.

In [ ]:
# Hint: lf.get_provider_parameters().groupby("category_key")


### AC Solvers

OpenLoadFlow provides three AC solver types:

- **Newton-Raphson** (default): classic full Jacobian solver
- **Fast Decoupled**: uses P-θ / Q-V decoupling for faster iterations
- **Newton-Krylov**: iterative linear solver, better for very large systems

**Task:** Print the available AC solver types from the provider parameters.

In [ ]:
# Hint: lf.get_provider_parameters("OpenLoadFlow").loc["acSolverType", "possible_values"]


### DC Solver

The DC approximation linearizes the power flow equations by assuming flat voltage magnitudes ($|V| = 1$ p.u.) and small angle differences ($\sin(\theta) \approx \theta$).

**Task:** Print the available DC approximation types.

In [ ]:
# Hint: lf.get_provider_parameters("OpenLoadFlow").loc["dcApproximationType", "possible_values"]


### Solver Comparison on MicroGrid T4

Compare all three AC solvers (Newton-Raphson, Fast Decoupled, Newton-Krylov) and the DC approximation on the MicroGrid T4 network.

**Task:** For each solver, run the power flow and compare: convergence status, iteration count, slack bus mismatch, bus voltage magnitudes, and bus voltage angles.

**Hints:**
- Use `lf.Parameters(provider_parameters={'acSolverType': 'NEWTON_RAPHSON'})` etc.
- Use `net.clone_variant()` to run each solver on a fresh copy
- Use `lf.run_dc()` for the DC case

In [ ]:
net = get_microgrid_t4()

solvers = {
    "NR": lf.Parameters(provider_parameters={"acSolverType": "NEWTON_RAPHSON"}),
    "FD": lf.Parameters(provider_parameters={"acSolverType": "FAST_DECOUPLED"}),
    "NK": lf.Parameters(provider_parameters={"acSolverType": "NEWTON_KRYLOV"}),
    "DC": lf.Parameters()
}

for p in solvers.values():
    p.hvdc_ac_emulation = False

# Create variants
base = net.get_working_variant_id()
for k in solvers:
    net.clone_variant(base, f"lf_{k}")

# Run load flows
results = {}
lf_results = {}

for name, params in solvers.items():
    net.set_working_variant(f"lf_{name}")
    # TODO: handle the DC case separately using lf.run_dc(), and AC cases with lf.run_ac()
    buses = net.get_buses()
    results[name] = {"v_mag": buses.v_mag.copy(), "v_angle": buses.v_angle.copy()}
    lf_results[name] = res

# Load-flow summary
df_loadflow_results = pd.DataFrame({
    name: {
        "type": "DC" if name == "DC" else "AC",
        "status": str(res.status),
        "iterations": getattr(res, "iteration_count", None),
        "slack_P_MW": (
            res.slack_bus_results[0].active_power_mismatch
            if getattr(res, "slack_bus_results", None) else None
        )
    }
    for name, res in lf_results.items()
}).T

# Bus voltage magnitudes
df_v_mag = pd.DataFrame({name: r["v_mag"] for name, r in results.items()}).round(2)
df_v_mag_fmt = df_v_mag.copy()
df_v_mag_fmt.index = df_v_mag_fmt.index.astype(str).str[:5]

# Bus voltage angles
df_v_angle = pd.DataFrame({name: r["v_angle"] for name, r in results.items()}).round(2)
df_v_angle_fmt = df_v_angle.copy()
df_v_angle_fmt.index = df_v_angle_fmt.index.astype(str).str[:5]

print("Load Flow Results:")
display(df_loadflow_results)
print("\nBus voltage [kV] results by solver:")
display(df_v_mag_fmt)
print("\nBus angle [deg] results by solver:")
display(df_v_angle_fmt)

### Convergence Criteria Comparison

The Newton-Raphson solver uses stopping criteria to decide when the solution has converged. Tighter tolerances produce more accurate results but require more iterations.

**Task:** Compare a tight (0.01) vs. loose (1.0) mismatch tolerance across all equation types. Observe the difference in iteration count.

**Hints:**
- Set `newtonRaphsonStoppingCriteriaType` to `PER_EQUATION_TYPE_CRITERIA`
- Use `maxActivePowerMismatch`, `maxReactivePowerMismatch`, etc.

In [ ]:
net = get_microgrid_t4()

PARAMS_tight = lf.Parameters(provider_parameters={
    "acSolverType": "NEWTON_RAPHSON",
    "newtonRaphsonStoppingCriteriaType": "PER_EQUATION_TYPE_CRITERIA",
    "maxActivePowerMismatch": "1e-2",
    "maxReactivePowerMismatch": "1e-2",
    "maxVoltageMismatch": "1e-2",
    "maxAngleMismatch": "1e-2",
    "maxRatioMismatch": "1e-2",
    "maxSusceptanceMismatch": "1e-2",
    "maxNewtonRaphsonIterations": "50"
})

# TODO: define PARAMS_loose with all tolerances set to 1.0

cases = {"0.01MW tolerance": PARAMS_tight, "1MW tolerance": PARAMS_loose}

base = net.get_working_variant_id()
for k in cases:
    net.clone_variant(base, f"conv_{k}")

lf_results = {}
for name, params in cases.items():
    net.set_working_variant(f"conv_{name}")
    res = lf.run_ac(net, params)[0]
    lf_results[name] = res

df_conv = pd.DataFrame({
    name: {"status": str(r.status), "iterations": r.iteration_count}
    for name, r in lf_results.items()
}).T

display(df_conv)

### Newton-Raphson vs Newton-Krylov Benchmark on RTE 6515

For large networks (thousands of buses), the **Newton-Krylov** solver avoids the full Jacobian factorization by using an iterative (GMRES-based) linear solver, which can be faster.

**Task:** Benchmark both solvers on the 6,515-bus RTE network. Use `get_rte_6515()` and measure execution time over multiple runs.

**Hint:** Use `time.perf_counter()` and `component_mode=lf.ComponentMode.MAIN_CONNECTED`.

In [ ]:
net = get_rte_6515()

def params(kind, solver):
    return lf.Parameters(
        voltage_init_mode=lf.VoltageInitMode.UNIFORM_VALUES,
        distributed_slack=(kind == "STANDARD"),
        use_reactive_limits=(kind == "STANDARD"),
        phase_shifter_regulation_on=False,
        transformer_voltage_control_on=False,
        component_mode=lf.ComponentMode.MAIN_CONNECTED,
        provider_parameters={"acSolverType": solver},
    )

def bench(kind, solver, runs=10, warmup=2):
    p = params(kind, solver)
    for _ in range(warmup):
        lf.run_ac(net, parameters=p)
    times = []
    last = None
    for _ in range(runs):
        t0 = time.perf_counter()
        last = lf.run_ac(net, parameters=p)
        times.append((time.perf_counter() - t0) * 1000)
    cr = last[0]
    # TODO: print solver name, status, iteration count, and timing statistics (best, median, mean)

for kind in ("BASIC", "STANDARD"):
    bench(kind, "NEWTON_RAPHSON", runs=10, warmup=2)
    bench(kind, "NEWTON_KRYLOV", runs=10, warmup=2)

---

## Power Flow Starting Vector

The initial guess (starting vector) for Newton-Raphson significantly impacts convergence speed and reliability.

### Available Initialization Modes

- **UNIFORM_VALUES**: flat start ($|V| = 1$ p.u., $\theta = 0$)
- **VOLTAGE_MAGNITUDE**: uses generator voltage setpoints as initial magnitude guesses
- **DC_VALUES**: initializes angles from a DC power flow solution
- **FULL_VOLTAGE**: combines DC_VALUES + VOLTAGE_MAGNITUDE — generally the best
- **PREVIOUS_VALUES**: reuses the state from a prior solve

**Task:** Compare all initialization modes on MicroGrid T4. For each mode, extract the initial mismatch norm $|f(x)|$ from the report and the total iteration count.

**Hints:**
- Use `rp.ReportNode()` to capture the solver report
- Parse the report string to find `Newton-Raphson norm |f(x)|=`
- For PREVIOUS_VALUES, run a first solve with UNIFORM_VALUES, then re-solve with PREVIOUS_VALUES

In [ ]:
net = get_microgrid_t4()

START_MODES = {
    "UNIFORM": lf.Parameters(voltage_init_mode=lf.VoltageInitMode.UNIFORM_VALUES),
    "VOLTAGE_MAG": lf.Parameters(
        provider_parameters={"voltageInitModeOverride": "VOLTAGE_MAGNITUDE"}
    ),
    "DC_VALUES": lf.Parameters(voltage_init_mode=lf.VoltageInitMode.DC_VALUES),
    "FULL_VOLT": lf.Parameters(
        provider_parameters={"voltageInitModeOverride": "FULL_VOLTAGE"}
    ),
}

base = net.get_working_variant_id()
for k in list(START_MODES) + ["PREVIOUS"]:
    net.clone_variant(base, f"sv_{k}")

rows = {}

for name, init in START_MODES.items():
    net.set_working_variant(f"sv_{name}")
    p = lf.Parameters(
        voltage_init_mode=init.voltage_init_mode,
        provider_parameters={
            **getattr(init, "provider_parameters", {}),
            "acSolverType": "NEWTON_RAPHSON",
            "reportedFeatures": "NEWTON_RAPHSON_LOAD_FLOW",
        },
    )
    p.hvdc_ac_emulation = False
    report = rp.ReportNode()
    res = lf.run_ac(net, p, report_node=report)[0]
    fx0 = None
    for line in str(report).splitlines():
        if "Newton-Raphson norm |f(x)|=" in line:
            fx0 = float(line.split("=")[-1])
            break
    rows[name] = {"|f(x)| initial": fx0, "iterations": res.iteration_count}

# TODO: add the PREVIOUS_VALUES case — first solve with UNIFORM_VALUES,
# then re-solve with lf.VoltageInitMode.PREVIOUS_VALUES and extract |f(x)|

pd.DataFrame(rows).T

---

## Generator Reactive Capability Curves

Real generators have operating limits defined by a **P-Q capability diagram**: the feasible region of active power (P) and reactive power (Q) that the machine can deliver simultaneously.

When `use_reactive_limits=True`, the solver enforces these curves by switching PV buses to PQ when a generator hits its reactive limit.

**Task:** Load MicroGrid T4, inspect the generators, get the reactive capability curve points, run a load flow, and plot the curve for generator BE-G1.

**Hints:**
- `net.get_generators()` shows generator data
- `net.get_reactive_capability_curve_points()` returns the P-Q diagram points
- `plot_curve(net, gen_id)` plots the capability curve with the operating point

In [ ]:
# Hint: net.get_generators(), net.get_reactive_capability_curve_points()


In [ ]:
# Hint: g_curve = '3a3b27be-b18b-4385-b557-6735d733baf0'  (BE-G1)
# Run lf.run_ac(), then plot_curve(net, g_curve)


---

## Slack Buses, PV Nodes and Slack Distribution

In power flow, one bus must absorb the active power mismatch between total generation and total load+losses — this is the **slack bus**.

### Slack Bus Selection

OpenLoadFlow offers multiple strategies: CGMES-provided, First bus, Most meshed, Largest generator, by Name/ID, and Country filter.

**Task:** Compare all slack bus selection strategies on MicroGrid T4. Create a summary table with status, slack bus ID, and mismatch.

**Hints:**
- `read_slack_bus=True` uses the CGMES extension
- `provider_parameters={'slackBusSelectionMode': 'MOST_MESHED'}` etc.

In [ ]:
tmp = get_microgrid_t4()
slack_bus_from_ext = tmp.get_buses().index[0]

def run_slack_case(case_name, params):
    net = get_microgrid_t4()
    res = lf.run_ac(net, params)[0]
    return {
        "case": case_name,
        "status": str(res.status),
        "slack_bus": ", ".join([s.id for s in res.slack_bus_results]),
        "slack_mismatch_MW": float(res.slack_bus_results[0].active_power_mismatch),
        "distributed_P_MW": float(getattr(res, "distributed_active_power", 0.0)),
    }

cases = []

# 1) Use CGMES-provided slack bus
cases.append(run_slack_case(
    "CGMES slack", lf.Parameters(read_slack_bus=True, distributed_slack=False)))

# 2) First bus
cases.append(run_slack_case(
    "First bus", lf.Parameters(read_slack_bus=False, distributed_slack=False,
                               provider_parameters={"slackBusSelectionMode": "FIRST"})))

# 3) Most meshed bus
cases.append(run_slack_case(
    "Most meshed", lf.Parameters(read_slack_bus=False, distributed_slack=False,
                                  provider_parameters={
                                      "slackBusSelectionMode": "MOST_MESHED",
                                      "mostMeshedSlackBusSelectorMaxNominalVoltagePercentile": "95"
                                  })))

# 4) Largest generator
cases.append(run_slack_case(
    "Largest generator", lf.Parameters(read_slack_bus=False, distributed_slack=False,
                                        provider_parameters={"slackBusSelectionMode": "LARGEST_GENERATOR"})))

# 5) By name/ID
cases.append(run_slack_case(
    "Name/ID", lf.Parameters(read_slack_bus=False, distributed_slack=False,
                              provider_parameters={
                                  "slackBusSelectionMode": "NAME",
                                  "slackBusesIds": str(slack_bus_from_ext)
                              })))

# TODO: add case 6 — Country filter using slackBusCountryFilter: "NL"

pd.DataFrame(cases).set_index("case")

### Reference Bus Selection

The **reference bus** sets the angle origin ($\theta = 0°$). By default it coincides with the slack bus, but OpenLoadFlow allows setting it independently.

**Task:** Compare `FIRST_SLACK` vs `GENERATOR_REFERENCE_PRIORITY` reference bus selection modes.

In [ ]:
def run_reference_bus_case(case_name, reference_mode):
    net = get_microgrid_t4()
    params = lf.Parameters(
        provider_parameters={
            "referenceBusSelectionMode": reference_mode,
            "writeReferenceTerminals": "true"
        }
    )
    res = lf.run_ac(net, params)[0]
    return {
        "case": case_name,
        "referenceBusSelectionMode": reference_mode,
        "reference_bus": res.reference_bus_id,
        "slack_buses": ", ".join([s.id for s in res.slack_bus_results])
    }

pd.DataFrame([
    run_reference_bus_case("Reference = FIRST_SLACK", "FIRST_SLACK"),
    # TODO: add GENERATOR_REFERENCE_PRIORITY case
]).set_index("case")

### PV Buses

A **PV bus** is a bus where a generator controls the voltage magnitude to a setpoint ($V_{target}$). The solver determines the reactive power $Q$ needed to maintain the target voltage.

**Task:** Run a load flow on MicroGrid T4 and inspect generators with voltage regulation enabled. Show the PV buses and their controlling generators.

In [ ]:
net = get_microgrid_t4()
lf.run_ac(net, lf.Parameters())

gens = net.get_generators(attributes=["name", "bus_id", "voltage_level_id", "p", "q",
                                       "voltage_regulator_on", "target_v", "target_p"])
gens_vctrl = gens[gens["voltage_regulator_on"] == True].copy()
display(gens_vctrl[["name", "bus_id", "target_v", "p", "q"]].sort_values(["bus_id", "name"]))

# TODO: group PV buses by bus_id showing controllers and target voltages


### Active Power Mismatch Distribution

When `distributed_slack=True`, the active power imbalance is shared among generators. Two strategies:

- **Participation factor**: proportional to `participation_factor` parameter
- **Pmax**: proportional to maximum active power capacity

**Task:** Compare both distribution strategies by adding 20 MW extra load and observing how generators share the mismatch.

**Hints:**
- Use `net.get_extensions('activePowerControl')` to set participation factors
- Use `BalanceType.PROPORTIONAL_TO_GENERATION_PARTICIPATION_FACTOR` and `PROPORTIONAL_TO_GENERATION_P_MAX`

In [ ]:
def run_mismatch_case(case_name, balance_type):
    net = get_microgrid_t4()
    g60 = "9c3b8f97-7972-477d-9dc8-87365cc0ad0e"  # NL-G1
    g30 = "2844585c-0d35-488d-a449-685bcd57afbf"  # NL-G2
    g10 = "1dc9afba-23b5-41a0-8540-b479ed8baf4b"  # NL-G3

    apc = net.get_extensions("activePowerControl")
    apc["participation_factor"] = 0.0
    apc.loc[[g60, g30, g10], "participation_factor"] = [0.6, 0.3, 0.1]
    apc.loc[[g60, g30, g10], "participate"] = True
    net.update_extensions("activePowerControl", apc)

    load_id = net.get_loads().index[0]
    p0 = float(net.get_loads().loc[load_id, "p0"])
    net.update_loads(id=load_id, p0=p0 + 20.0)

    params = lf.Parameters(distributed_slack=True, balance_type=balance_type)
    res = lf.run_ac(net, params)[0]

    g = net.get_generators(attributes=["name", "p", "target_p", "max_p"]).copy()
    apc = net.get_extensions("activePowerControl")[["participate", "participation_factor"]]
    g = g.join(apc, how="left")
    g["case"] = case_name
    return g[["case", "name", "participate", "participation_factor", "max_p", "p", "target_p"]], res

t1, r1 = run_mismatch_case(
    "PF weights (participation_factor)",
    lf.BalanceType.PROPORTIONAL_TO_GENERATION_PARTICIPATION_FACTOR
)
display(t1.sort_values(["case", "participation_factor"], ascending=[True, False]))
print("Case 1 distributed P (MW):", float(r1.distributed_active_power),
      " slack mismatch (MW):", float(r1.slack_bus_results[0].active_power_mismatch))

# TODO: run second case with PROPORTIONAL_TO_GENERATION_P_MAX and display results

### Distributed Slack Implementation

Compare `distributed_slack=False` (all mismatch on slack) vs `distributed_slack=True` (shared).

**Task:** Add 20 MW extra load and compare both modes. Show the slack mismatch and distributed power.

In [ ]:
net = get_microgrid_t4()

display(net.get_extensions("activePowerControl")[["participate", "participation_factor"]])

load_id = net.get_loads().index[0]

def run_case(label, distributed):
    net2 = get_microgrid_t4()
    net2.update_loads(id=load_id, p0=float(net2.get_loads().loc[load_id, "p0"]) + 20.0)
    res = lf.run_ac(net2, lf.Parameters(
        read_slack_bus=True,
        distributed_slack=distributed,
        balance_type=lf.BalanceType.PROPORTIONAL_TO_GENERATION_PARTICIPATION_FACTOR
    ))[0]
    return pd.DataFrame([{
        "case": label,
        "slack_buses": ", ".join([s.id for s in res.slack_bus_results]),
        "slack_mismatch_MW": float(res.slack_bus_results[0].active_power_mismatch),
        "distributed_P_MW": float(res.distributed_active_power),
    }])

display(pd.concat([
    run_case("distributed_slack = False", False),
    # TODO: add distributed_slack = True case
], ignore_index=True))

### Slack Distribution on Loads (ReliCapGrid)

The mismatch can also be distributed on **loads** using `PROPORTIONAL_TO_LOAD` balance type.

**Task:** Test load slack distribution on ReliCapGrid Belgovia with and without constant power factor.

**Hints:**
- `BalanceType.PROPORTIONAL_TO_LOAD`
- `provider_parameters={'loadPowerFactorConstant': 'true'/'false'}`

In [ ]:
# Case A: PF not forced
net = get_relicapgrid_belgovia()
load_id = net.get_loads().index[0]
net.update_loads(id=load_id, p0=float(net.get_loads().loc[load_id, "p0"]) + 20.0)

params = lf.Parameters(
    distributed_slack=True,
    balance_type=lf.BalanceType.PROPORTIONAL_TO_LOAD,
    provider_parameters={"loadPowerFactorConstant": "false"},
)
lf.run_ac(net, params)
print("\n--- Slack on loads (PF not forced) ---")
display(net.get_loads()[["p0", "q0", "p", "q"]])

# TODO: add Case B with loadPowerFactorConstant = "true" and compare

### Slack Distribution Failure Behavior

What happens when generators cannot absorb the full mismatch? Three failure behaviors:

- **FAIL**: the power flow fails entirely
- **LEAVE_ON_SLACK_BUS**: leave the residual mismatch on the slack bus
- **DISTRIBUTE_ON_REFERENCE_GENERATOR**: force the reference generator to absorb it

**Task:** Freeze generator limits (set min_p = max_p = target_p) to prevent redistribution, add 10 MW load, and test all three failure behaviors.

In [ ]:
net = get_microgrid_t4()
base = net.get_working_variant_id()

# Freeze generators at target_p (no room to move)
g = net.get_generators()[["target_p", "min_p", "max_p"]].copy()
g["min_p"] = g["target_p"]
g["max_p"] = g["target_p"]
net.update_generators(g[["min_p", "max_p"]])

net.clone_variant(base, "FAIL")
net.clone_variant(base, "LEAVE_ON_SLACK_BUS")
net.clone_variant(base, "DISTRIBUTE_ON_REFERENCE_GENERATOR")

# FAIL
net.set_working_variant("FAIL")
load_id = net.get_loads().index[0]
net.update_loads(id=load_id, p0=float(net.get_loads().loc[load_id, "p0"]) + 10.0)
params = lf.Parameters(distributed_slack=True,
                        balance_type=lf.BalanceType.PROPORTIONAL_TO_GENERATION_P_MAX,
                        provider_parameters={"slackDistributionFailureBehavior": "FAIL"})
res = lf.run_ac(net, params)[0]
print("--- FAIL ---")
print("status:", res.status)
print("slack mismatch (MW):", float(res.slack_bus_results[0].active_power_mismatch))

# LEAVE_ON_SLACK_BUS
net.set_working_variant("LEAVE_ON_SLACK_BUS")
load_id = net.get_loads().index[0]
net.update_loads(id=load_id, p0=float(net.get_loads().loc[load_id, "p0"]) + 10.0)
params = lf.Parameters(distributed_slack=True,
                        balance_type=lf.BalanceType.PROPORTIONAL_TO_GENERATION_P_MAX,
                        provider_parameters={"slackDistributionFailureBehavior": "LEAVE_ON_SLACK_BUS"})
res = lf.run_ac(net, params)[0]
print("\n--- LEAVE_ON_SLACK_BUS ---")
print("status:", res.status)
print("slack mismatch (MW):", float(res.slack_bus_results[0].active_power_mismatch))

# TODO: add DISTRIBUTE_ON_REFERENCE_GENERATOR case using the same pattern

### Reactive Power Sharing on a Common PV Bus

When multiple generators regulate the same bus voltage, their reactive power contributions must be coordinated.

**Task:** Compare Q sharing when both generators regulate vs only one regulates the same bus.

In [ ]:
net = get_microgrid_t4()

gens = net.get_generators()
pv_bus = gens.bus_id.value_counts().idxmax()
pv_gens = gens[gens.bus_id == pv_bus].index.tolist()
g1, g2 = pv_gens[:2]

rows = []

# Case 1: both regulate (no limits)
net.update_generators(pd.DataFrame({"voltage_regulator_on": True}, index=[g1, g2]))
params = lf.Parameters(use_reactive_limits=False,
                        provider_parameters={"acSolverType": "NEWTON_RAPHSON"})
params.hvdc_ac_emulation = False
lf.run_ac(net, params)
df = net.get_generators()[["q", "target_q"]].loc[[g1, g2]]
df["case"] = "both regulating (no limits)"
rows.append(df)

# TODO: add Case 2 — set g1 voltage_regulator_on=False (single regulating) and compare Q values

display(pd.concat(rows))

### Q_EQUAL_PROPORTION Sharing with Modified Qmax

The `Q_EQUAL_PROPORTION` mode distributes Q proportionally to each generator's reactive capability range ($Q_{max} - Q_{min}$).

**Task:** Modify one generator's $Q_{max}$ and observe how the Q sharing changes.

In [ ]:
net = get_microgrid_t4()

crc = net.get_extensions("coordinatedReactiveControl")
net.remove_extensions("coordinatedReactiveControl", ids=list(crc.index))

gens = net.get_generators()
bus_id = gens.bus_id.value_counts().idxmax()
g1, g2 = gens[gens.bus_id == bus_id].index[:2]

net.update_generators(pd.DataFrame({"voltage_regulator_on": True}, index=[g1, g2]))
net.update_generators(id=[g1, g2], target_v=[17, 17])
net.update_generators(id=[g1], max_q=[300.0], min_q=[-100])

params = lf.Parameters(use_reactive_limits=True,
                        provider_parameters={"reactivePowerDispatchMode": "Q_EQUAL_PROPORTION"})
lf.run_ac(net, params)

res = net.get_generators()[["name", "bus_id", "q", "min_q", "max_q"]].loc[[g1, g2]].copy()
# TODO: compute q_share = q / q.sum() to see the proportional sharing
res

---

## Convergence Reporting

PowSyBl provides detailed **iteration-by-iteration reporting** via `ReportNode`. This lets you inspect the convergence behavior: initial mismatch $|f(x)|$, how it decreases at each iteration, and any outer-loop adjustments.

**Task:** Demonstrate four convergence scenarios:
1. **Fast convergence** (~5 iterations): Full voltage init + no reactive limits
2. **Slow convergence** (~15 iterations): Flat start + low state vector scaling
3. **Bad convergence** (~10 iterations): Reactive limits + PV-to-PQ switching
4. **Divergence**: Unrealistic generator voltage setpoint

**Hints:**
- Use `rp.ReportNode('demo', 'demo')` to capture report
- `provider_parameters={'reportedFeatures': 'NEWTON_RAPHSON_LOAD_FLOW'}` to enable reporting
- For slow convergence: `stateVectorScalingMode: MAX_VOLTAGE_CHANGE`
- For divergence: set a large `target_v` on a generator

In [ ]:
PARAMS_REPORTED_LF_FAST_CV = lf.Parameters(
    use_reactive_limits=False,
    provider_parameters={
        'referenceBusSelectionMode': 'GENERATOR_REFERENCE_PRIORITY',
        "reportedFeatures": "NEWTON_RAPHSON_LOAD_FLOW",
        "voltageInitModeOverride": "FULL_VOLTAGE",
    }
)

net_microgrid = get_microgrid_t4()
# TODO: create a ReportNode, run lf.run_ac with report_node=report, and print the report

In [ ]:
PARAMS_REPORTED_LF_SLOW_CV = lf.Parameters(
    use_reactive_limits=False,
    provider_parameters={
        "reportedFeatures": "NEWTON_RAPHSON_LOAD_FLOW",
        "voltageInitModeOverride": "NONE",
        "stateVectorScalingMode": "MAX_VOLTAGE_CHANGE",
        "maxVoltageChangeStateVectorScalingMaxDv": "0.01",
    }
)

# TODO: create a ReportNode, run lf.run_ac on net_microgrid, and print the report

In [ ]:
PARAMS_REPORTED_LF_BAD_CV = lf.Parameters(
    use_reactive_limits=True,
    provider_parameters={
        "reportedFeatures": "NEWTON_RAPHSON_LOAD_FLOW",
        "voltageInitModeOverride": "NONE",
    }
)

# TODO: create a ReportNode, run lf.run_ac on net_microgrid, and print the report

In [ ]:
gen = net_microgrid.get_generators()
gen.loc[gen["name"] == "NL-G1", "target_v"] = 25  # kV instead of 16kV
net_microgrid.update_generators(gen[["target_v"]])

# TODO: create a ReportNode, run lf.run_ac on net_microgrid with PARAMS_REPORTED_LF_BAD_CV, and print the report

---

## Handling Bad Convergence

When a power flow diverges, two key recovery techniques:

1. **Full voltage initialization** (`FULL_VOLTAGE`): better starting point via DC pre-solve
2. **State vector scaling** (`MAX_VOLTAGE_CHANGE`): limits voltage change per Newton iteration

**Task:** Create a divergent case (unrealistic generator voltage setpoint), then fix it using both techniques combined.

**Hints:**
- Use variants: clone base to 'BAD' and 'FIXED'
- Set NL-G1 target_v to 24 kV (vs normal ~16 kV)
- BAD case: `voltageInitModeOverride: NONE`
- FIXED case: `voltageInitModeOverride: FULL_VOLTAGE` + `stateVectorScalingMode: MAX_VOLTAGE_CHANGE`

In [ ]:
net = get_microgrid_t4()

base = net.get_working_variant_id()
net.clone_variant(base, "BAD")
net.clone_variant(base, "FIXED")

GEN_NAME = "NL-G1"
NEW_TARGET_V = 24  # kV

# BASE
net.set_working_variant(base)
g0 = net.get_generators()
gid = g0.index[g0["name"].eq(GEN_NAME)][0]
p0 = lf.Parameters()
r0 = lf.run_ac(net, p0)[0]
print("BASE:", r0.status, "iters:", getattr(r0, "iteration_count", None))

# Diverging case
net.set_working_variant("BAD")
g = net.get_generators()
g.loc[gid, "target_v"] = NEW_TARGET_V
net.update_generators(g[["target_v"]])

report_bad = rp.ReportNode("BAD", "Bad target voltage")
p_bad = lf.Parameters(provider_parameters={
    "voltageInitModeOverride": "NONE",
    "acSolverType": "NEWTON_RAPHSON",
    "reportedFeatures": "NEWTON_RAPHSON_LOAD_FLOW"
})
r_bad = lf.run_ac(net, p_bad, report_node=report_bad)[0]
print("\n--- BAD report ---")
print(report_bad)

# TODO: add FIXED case — set working variant to "FIXED", update generator target_v,
# then run with voltageInitModeOverride="FULL_VOLTAGE" and stateVectorScalingMode="MAX_VOLTAGE_CHANGE"

---

## Electrical Islands

PowSyBl distinguishes:

- **Connected Components**: groups of buses linked by any path (AC or DC)
- **Synchronous Components**: groups of buses linked only by AC paths (same frequency)

Two buses connected through an HVDC link are in the same connected component but in different synchronous components.

**Task:** Use the 6-bus network to explore connected/synchronous components. Then remove AC lines going to a bus to create an AC island connected through the HVDC link. Verify the component structure changes.

**Hints:**
- `net.get_bus_breaker_view_buses(attributes=['connected_component', 'synchronous_component'])`
- `net.remove_elements(line_ids)` to remove AC lines
- Use `component_mode=lf.ComponentMode.ALL_CONNECTED` to solve all components

In [ ]:
# Hint: get_six_buses(), then net.get_bus_breaker_view_buses(attributes=[...])
# Group by connected_component and synchronous_component


In [ ]:
hvdc_bus = "SO_poste_0"
print("Selected HVDC AC bus:", hvdc_bus)

lines = net_six_buses.get_lines()
lines_to_remove = lines[
    (lines.bus1_id == hvdc_bus) | (lines.bus2_id == hvdc_bus)
].index.tolist()
print("Removing AC lines:", lines_to_remove)

net_six_buses.remove_elements(lines_to_remove)

hvdc = net_six_buses.get_hvdc_lines()
net_six_buses.update_hvdc_lines(id=hvdc.index.tolist(), target_p=[250.0, 250.0])

b2 = net_six_buses.get_bus_breaker_view_buses(
    attributes=["name", "connected_component", "synchronous_component"]
)
b2 = b2[(b2.connected_component >= 0) & (b2.synchronous_component >= 0)]
display(b2.groupby(["connected_component", "synchronous_component"]).size().reset_index(name="bus_count"))

# TODO: run AC load flow with ComponentMode.ALL_CONNECTED, reportedFeatures, and print the report

---

## Inspecting Load Flow Results

After a successful power flow, you can inspect all computed quantities: bus voltages (magnitude and angle), branch flows (P, Q, I at both ends), transformer tap positions, shunt compensator states, and generator/load operating points.

**Task:** Run a load flow on MicroGrid T4 and display:
1. Bus voltages (magnitude and angle)
2. Line flows (P, Q, I at both ends) and losses
3. 2-winding transformer flows
4. 3-winding transformer flows
5. Tap changer positions
6. Shunt compensator states
7. Generator operating points
8. Load operating points
9. SVC operating points

In [ ]:
# Hint: get_microgrid_t4(), run_lf(), then
# net.get_bus_breaker_view_buses(attributes=["name", "v_mag", "v_angle", ...])


In [ ]:
# Hint: net.get_lines(attributes=["name", "p1", "q1", "i1", "p2", "q2", "i2", ...])
# Compute losses: p_losses = p1 + p2


In [ ]:
# Hint: net.get_2_windings_transformers(attributes=[...])
# and net.get_3_windings_transformers(attributes=[...])


In [ ]:
# Display: get_ratio_tap_changers(), get_phase_tap_changers()
# get_shunt_compensators(), get_generators(), get_loads()
# get_static_var_compensators()


---

## FACTS and HVDC Models

PowSyBl supports **Static VAR Compensators** (SVCs) for reactive power support and **VSC-HVDC** links for controllable active power transfer.

### Static VAR Compensators (SVCs)

**Task:** Load MicroGrid T4, run a load flow, and inspect the SVC data. Explore the network around the SVC voltage level.

In [ ]:
# Hint: get_microgrid_t4(), run_lf(), net.get_static_var_compensators()


### HVDC (Reduced Model)

Two models: **Reduced** (simplified) and **Detailed** (full DC circuit, WIP).

**Task:** Load the 6-bus network, inspect the HVDC lines and VSC converter stations. Change the active power setpoint to 100 MW and enable voltage regulation at 406 kV. Run the load flow and visualize.

**Hints:**
- `net.get_hvdc_lines()` and `net.get_vsc_converter_stations()`
- `net.update_hvdc_lines(id='HVDC1', target_p=100.)`
- `net.update_vsc_converter_stations(id=..., voltage_regulator_on=True)`

In [ ]:
net_six_buses = get_six_buses()
run_lf(net_six_buses)

display(net_six_buses.get_hvdc_lines())
display(net_six_buses.get_vsc_converter_stations())

# TODO: update HVDC lines target_p to 100 MW, enable voltage_regulator_on for all VSC stations,
# re-run load flow, and display updated hvdc_lines and vsc_converter_stations

In [ ]:
# Hint: plot_curve(net_six_buses, 'SOO1_SOO1_HVDC1')


---

## Series and Shunt Compensators

### Series Compensators

In CGMES, `cim:SeriesCompensator` elements are modelled as lines with negative reactance ($X < 0$).

**Task:** Load ReliCapGrid Belgovia and identify series compensators by filtering lines with `CGMES.originalClass`.

In [ ]:
# Hint: get_relicapgrid_belgovia(), net.get_lines(attributes=["name", "r", "x", "CGMES.originalClass"])


### Shunt Compensators

PowSyBl supports both Linear and Non-Linear shunt compensators.

**Task:** On MicroGrid T4:
1. List all shunt compensators and their types
2. Run a load flow and display their operating points
3. Switch off BE-S2 (section_count=0) and re-run
4. Change BE-S2 to section_count=3 and re-run

**Hint:** BE-S2 mRID: `002b0a40-3957-46db-b84a-30420083558f`

In [ ]:
net_microgrid_t4 = get_microgrid_t4()
display(net_microgrid_t4.get_shunt_compensators()[['name', 'model_type', 'max_section_count']])

run_lf(net_microgrid_t4)
display(net_microgrid_t4.get_shunt_compensators())

# Linear and non-linear sections
display(net_microgrid_t4.get_linear_shunt_compensator_sections())
display(net_microgrid_t4.get_non_linear_shunt_compensator_sections())

# Switch off BE-S2 (section_count=0)
be_s2 = '002b0a40-3957-46db-b84a-30420083558f'
net_microgrid_t4.update_shunt_compensators(id=be_s2, section_count=0)
run_lf(net_microgrid_t4)
print("\nBE-S2 switched off (section_count=0):")
display(net_microgrid_t4.get_shunt_compensators().loc[[be_s2]][
    ['name', 'model_type', 'max_section_count', 'section_count', 'p', 'q', 'i']
])

# TODO: set BE-S2 to section_count=3, re-run load flow, and display its operating point

---

## Tap Changers for Transformers

- **Ratio tap changers**: control voltage magnitude by adjusting the turns ratio $\rho$
- **Phase tap changers**: control active power flow by adjusting the phase angle $\alpha$

### Phase Tap Changers

**Task:** On MicroGrid T4:
1. Inspect the phase tap changer on BE-TR2_1
2. View its tap steps
3. Change regulation_value from -65 MW to -20 MW
4. Change tap from 10 to 12
5. Modify step attributes (angle and ratio)

**Hint:** BE-TR2_1 mRID: `a708c3bc-465d-4fe7-b6ef-6fa6408a62b0`

In [ ]:
net_microgrid_t4 = get_microgrid_t4()

# BE-TR2_1 phase tap changer
ptc_id = 'a708c3bc-465d-4fe7-b6ef-6fa6408a62b0'
display(net_microgrid_t4.get_phase_tap_changers().loc[[ptc_id]][
    ["low_tap", "high_tap", "tap", "solved_tap_position", "oltc", "regulating",
     "regulation_mode", "regulation_value", "regulating_bus_id", "target_deadband"]
])

# Phase tap changer steps
display(net_microgrid_t4.get_phase_tap_changer_steps().loc[[ptc_id]].sort_values(
    by=['id', 'position']
))

# Change regulation_value from -65 MW to -20 MW
net_microgrid_t4.update_phase_tap_changers(id=ptc_id, regulation_value=-20)
print("\nAfter regulation_value = -20:")
display(net_microgrid_t4.get_phase_tap_changers().loc[[ptc_id]][
    ["low_tap", "high_tap", "tap", "regulation_value"]
])

# Change tap from 10 to 12
net_microgrid_t4.update_phase_tap_changers(id=ptc_id, tap=12)
print("\nAfter tap = 12:")
display(net_microgrid_t4.get_phase_tap_changers().loc[[ptc_id]][
    ["low_tap", "high_tap", "tap"]
])

# TODO: modify step 2 attributes (rho=1.1, alpha=-12.3) using update_phase_tap_changer_steps and display

### Ratio Tap Changers

**Task:** Inspect the ratio tap changer on NL_TR2_3.

**Hint:** NL_TR2_3 mRID: `80016742-31b3-432a-b00a-300667a1e572`

In [ ]:
# Hint: net.get_ratio_tap_changers().loc[['80016742-31b3-432a-b00a-300667a1e572']]


### Only One Regulating Control Per Transformer

A transformer with both a phase and ratio tap changer can only have one actively regulating.

**Task:** Try to create a second regulating tap changer on BE-TR2_1 (which already has a phase tap changer) and observe the error.

In [ ]:
net_microgrid_t4 = get_microgrid_t4()

ptc_id = 'a708c3bc-465d-4fe7-b6ef-6fa6408a62b0'

ptc_df = pd.DataFrame.from_records(
    index='id',
    columns=['id', 'target_deadband', 'target_v', 'oltc', 'low_tap', 'tap', 'regulating', 'regulated_side'],
    data=[(ptc_id, 2, 200, True, 0, 1, False, 'TWO')]
)
steps_df = pd.DataFrame.from_records(
    index='id', columns=['id', 'b', 'g', 'r', 'x', 'rho'],
    data=[(ptc_id, 2, 2, 1, 1, 0.5), (ptc_id, 2, 2, 1, 1, 0.4), (ptc_id, 2, 2, 1, 1, 0.5)]
)
net_microgrid_t4.create_ratio_tap_changers(ptc_df, steps_df)
display(net_microgrid_t4.get_ratio_tap_changers().loc[[ptc_id]])

# TODO: try to activate the ratio tap changer (regulating=True) and catch the PyPowsyblError

---

## Zero-Impedance Elements

PowSyBl handles zero-impedance elements (bus couplers, breakers) via its topology processing layer. The IIDM model automatically merges zero-impedance branches into the bus topology.

**Task:** Load the 6-bus network and inspect the bus/breaker topology for the first voltage level.

**Hint:** `net.get_bus_breaker_topology(vl_id)`

In [ ]:
# Hint: get_six_buses(), net.get_voltage_levels().index[0]
# net.get_bus_breaker_topology(vl_id)
